On récup l'ancien code en enlevant les features qui paraisse gêner

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# ==============================================================================
# 1. FEATURE PRUNING (Ablation des indicateurs lents) & SAUVEGARDE V2
# ==============================================================================
print("1. Lancement du Feature Pruning...")
df_original = pd.read_csv('../data/processed/chip_chain_master_lstm.csv')

# On identifie et supprime TOUTES les colonnes qui contiennent "63" (ex: DMA_63d, Zscore_63d, etc.)
# Si tu as d'autres noms spécifiques pour tes features lentes, ajoute-les à la condition
cols_to_drop = [col for col in df_original.columns if '63' in str(col)]
print(f"Colonnes supprimées (Lagging Bias) : {cols_to_drop}")

df_v2 = df_original.drop(columns=cols_to_drop)

# Sauvegarde du nouveau Master Dataset V2
df_v2.to_csv('../data/processed/chip_chain_master_lstm2.csv', index=False)
print("✅ Dataset V2 sauvegardé sous 'chip_chain_master_lstm2.csv'")

# ==============================================================================
# 2. DATA ENGINEERING (Cible Lissée T+5 & Tenseurs 3D)
# ==============================================================================
print("\n2. Préparation des Tenseurs 3D...")
df = df_v2.copy()
df = pd.get_dummies(df, columns=['HMM_State'], prefix='Regime', dtype=int)

TARGET_ASSET = 'NVDA'

# Cible Lissée (Tendance des 5 prochains jours)
future_5d_return = df[TARGET_ASSET].shift(-5).rolling(window=5).mean()
df['Target_5D'] = np.where(future_5d_return > 0, 1, 0)
df = df.dropna().reset_index(drop=True)

# Masques et Scaling
train_mask = df['Dataset_Type'] == 'Train'
test_mask = df['Dataset_Type'] == 'Test'

features_cols = [col for col in df.columns if col not in ['Date', 'Dataset_Type', 'Target_5D']]

scaler = StandardScaler()
train_scaled = scaler.fit_transform(df.loc[train_mask, features_cols])
test_scaled = scaler.transform(df.loc[test_mask, features_cols])

X_scaled_full = np.vstack((train_scaled, test_scaled))
y_full = df['Target_5D'].values
type_full = df['Dataset_Type'].values

TIME_STEPS = 21
X_3d_train, y_3d_train, X_3d_test, y_3d_test = [], [], [], []

for i in range(len(X_scaled_full) - TIME_STEPS):
    window = X_scaled_full[i : i + TIME_STEPS]
    target_val = y_full[i + TIME_STEPS]
    day_type = type_full[i + TIME_STEPS] 
    
    if day_type == 'Train':
        X_3d_train.append(window)
        y_3d_train.append(target_val)
    else:
        X_3d_test.append(window)
        y_3d_test.append(target_val)

X_train_3d, y_train = np.array(X_3d_train), np.array(y_3d_train)
X_test_3d, y_test = np.array(X_3d_test), np.array(y_3d_test)

print(f"✅ Dimensions X_train_3d : {X_train_3d.shape} (Les features lentes ont disparu !)")

# ==============================================================================
# 3. ENTRAÎNEMENT DU LSTM V2 (Hyperparamètres optimisés)
# ==============================================================================
print("\n3. Entraînement du Modèle V2...")
# On fixe les seeds pour garantir la reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

model_v2 = Sequential()
model_v2.add(Input(shape=(TIME_STEPS, len(features_cols))))
model_v2.add(LSTM(units=32, activation='tanh', kernel_regularizer=l2(1e-6), recurrent_dropout=0.2))
model_v2.add(Dropout(0.5))
model_v2.add(Dense(1, activation='sigmoid'))

model_v2.compile(optimizer=Adam(learning_rate=5e-5), loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history = model_v2.fit(
    X_train_3d, y_train,
    epochs=100, batch_size=64, validation_split=0.15,
    callbacks=[early_stop], shuffle=False, verbose=0 # Verbose=0 pour ne pas polluer l'écran
)

_, test_acc = model_v2.evaluate(X_test_3d, y_test, verbose=0)
print(f"✅ Entraînement terminé ! Accuracy Test (T+5) : {test_acc * 100:.2f}%")
model_v2.save('../data/tensors/chip_chain_lstm_v2.keras')

# ==============================================================================
# 4. LE VERDICT : BACKTEST FINANCIER OUT-OF-SAMPLE
# ==============================================================================
print("\n4. Lancement du Backtest Quantitatif...")

test_dates = df.loc[test_mask, 'Date'].values
real_returns = df.loc[test_mask, TARGET_ASSET].values 
y_pred_probs = model_v2.predict(X_test_3d, verbose=0).flatten()

backtest_df = pd.DataFrame({'Return_Real': real_returns, 'Prob_Up': y_pred_probs}, index=test_dates)

# Stratégie : Achat si la proba est > 0.50
backtest_df['Position'] = np.where(backtest_df['Prob_Up'] > 0.50, 1, 0)
backtest_df['Position_Delayed'] = backtest_df['Position'].shift(1).fillna(0) # Prévention Look-Ahead Bias
backtest_df['Return_Strategy'] = backtest_df['Position_Delayed'] * backtest_df['Return_Real']

def calculate_metrics(returns):
    cum_returns = np.prod(1 + returns) - 1
    mean_ann = np.mean(returns) * 252
    vol_ann = np.std(returns) * np.sqrt(252)
    sharpe = mean_ann / vol_ann if vol_ann != 0 else 0
    
    equity_curve = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(equity_curve)
    drawdowns = (equity_curve - running_max) / running_max
    max_dd = np.min(drawdowns) if len(drawdowns) > 0 else 0
    return cum_returns, sharpe, max_dd

cum_bh, sharpe_bh, max_dd_bh = calculate_metrics(backtest_df['Return_Real'])
cum_strat, sharpe_strat, max_dd_strat = calculate_metrics(backtest_df['Return_Strategy'])

results_summary = pd.DataFrame({
    'Métrique': ['Rendement Cumulé', 'Sharpe Ratio (Ann.)', 'Max Drawdown'],
    'Buy & Hold (NVDA)': [f"{cum_bh * 100:.2f}%", f"{sharpe_bh:.2f}", f"{max_dd_bh * 100:.2f}%"],
    'LSTM V2 (Fast Features)': [f"{cum_strat * 100:.2f}%", f"{sharpe_strat:.2f}", f"{max_dd_strat * 100:.2f}%"]
})

print("\n" + "="*50)
print(" RÉSULTATS DU BACKTEST V2 (SANS BIAIS DE RETARD)")
print("="*50)
print(results_summary.to_string(index=False))

1. Lancement du Feature Pruning...
Colonnes supprimées (Lagging Bias) : ['DMA_63d_NVDA', 'DMA_63d_ASML', 'DMA_63d_TSM', 'DMA_63d_AMD']
✅ Dataset V2 sauvegardé sous 'chip_chain_master_lstm2.csv'

2. Préparation des Tenseurs 3D...
✅ Dimensions X_train_3d : (2804, 21, 47) (Les features lentes ont disparu !)

3. Entraînement du Modèle V2...
✅ Entraînement terminé ! Accuracy Test (T+5) : 54.31%

4. Lancement du Backtest Quantitatif...

 RÉSULTATS DU BACKTEST V2 (SANS BIAIS DE RETARD)
           Métrique Buy & Hold (NVDA) LSTM V2 (Fast Features)
   Rendement Cumulé           259.23%                 108.27%
Sharpe Ratio (Ann.)              1.20                    0.84
       Max Drawdown           -41.14%                 -46.32%


Bon on voit que le modèle est encore moins bon, il a finalement surement besoin d'indicateurs lents pour garder un cap, une vue gloabale